# Fine-Tune MDR — Evaluation on Colab

Runs eval on the full 31K test set using the trained LoRA adapter from Google Drive.
Much faster on Colab GPU than the DGX GB10.

In [ ]:
# Cell 1: GPU Check
import torch
assert torch.cuda.is_available(), "No GPU!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Install Dependencies
!pip install -q transformers>=4.57 peft>=0.18 datasets accelerate pyyaml rouge-score

In [ ]:
# Cell 3: Mount Drive + HF Token
import os
from google.colab import drive, userdata
drive.mount("/content/drive")

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception:
    pass

ADAPTER_DIR = "/content/drive/MyDrive/fine-tune-mdr/outputs/final_adapter"
print(f"Adapter: {os.listdir(ADAPTER_DIR)}")

In [ ]:
# Cell 4: Configuration
MODEL_NAME = "fdtn-ai/Foundation-Sec-8B-Instruct"
MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 512          # Increased from 256 — prevent truncation of longer responses
REPETITION_PENALTY = 1.1      # Discourage template repetition
BATCH_SIZE = 8
DATASET_NAME = "jason-oneal/mitre-stix-cve-exploitdb-dataset-alpaca-chatml-harmony"
DATASET_CONFIG = "chatml"
DATASET_SEED = 42
NUM_EXAMPLES = 500  # Set to None for full test set

In [ ]:
# Cell 5: Load Model + LoRA Adapter
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

torch.set_float32_matmul_precision("high")

print(f"Loading tokenizer from {ADAPTER_DIR}...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading base model {MODEL_NAME}...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    device_map={"": 0},
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 6: Load Test Dataset
from datasets import load_dataset

print("Loading dataset...")
raw = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
print(f"Total examples: {len(raw):,}")

# Split to get the same test set as training (5% test)
split = raw.train_test_split(test_size=0.10, seed=DATASET_SEED)
# The 10% was val+test; split again to get just test (half of 10% = 5%)
val_test = split["test"].train_test_split(test_size=0.5, seed=DATASET_SEED)
test_ds = val_test["test"]
print(f"Test set: {len(test_ds):,} examples")

# Apply chat template
columns = test_ds.column_names
if "text" not in columns:
    print("Applying chat template...")
    test_ds = test_ds.map(
        lambda ex: {"text": tokenizer.apply_chat_template(
            ex["messages"], tokenize=False, add_generation_prompt=False
        )},
        remove_columns=[c for c in columns if c != "text"],
        desc="Formatting",
    )
print(f"Test examples ready: {len(test_ds):,}")

In [ ]:
# Cell 7: Clear HF Cache
!rm -rf /root/.cache/huggingface/hub /root/.cache/huggingface/datasets
print("Cache cleared.")

In [ ]:
# Cell 8: Run Evaluation
import re
import time
from rouge_score import rouge_scorer

ARTIFACT_PREFIX_RE = re.compile(r"^(?:[>*]{1,2}\s*)+")

def postprocess_response(text):
    """Strip artifact prefixes (>, **, *) from model output."""
    lines = text.split("\n")
    cleaned = [ARTIFACT_PREFIX_RE.sub("", line) for line in lines]
    return "\n".join(cleaned).strip()

def extract_ground_truth(text):
    """Extract the ground truth assistant response from formatted text."""
    markers = [
        "<|assistant|>",
        "<|start_header_id|>assistant<|end_header_id|>",
        "### Assistant:",
    ]
    for marker in markers:
        idx = text.rfind(marker)
        if idx != -1:
            response = text[idx + len(marker):].strip()
            for token in ["<|eot_id|>", "</s>", "<|end_of_text|>"]:
                response = response.replace(token, "")
            return response.strip()
    return None

def build_prompt(text):
    """Extract prompt (everything up to and including the assistant marker)."""
    for marker in [
        "<|assistant|>",
        "<|start_header_id|>assistant<|end_header_id|>",
        "### Assistant:",
    ]:
        idx = text.rfind(marker)
        if idx != -1:
            return text[: idx + len(marker)]
    return text

# Subset if configured
eval_ds = test_ds
if NUM_EXAMPLES is not None:
    eval_ds = test_ds.select(range(min(NUM_EXAMPLES, len(test_ds))))

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
results = []
total = len(eval_ds)
start_time = time.time()

print(f"Evaluating {total} examples...\n")

for i in range(total):
    text = eval_ds[i]["text"]

    gt_text = extract_ground_truth(text)
    prompt = build_prompt(text)

    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH,
    ).to(model.device)

    with torch.no_grad():
        gen_kwargs = dict(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        if REPETITION_PENALTY != 1.0:
            gen_kwargs["repetition_penalty"] = REPETITION_PENALTY
        output_ids = model.generate(**gen_kwargs)

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    pred_text = postprocess_response(pred_text)

    # Compute ROUGE scores
    scores = scorer.score(gt_text, pred_text) if gt_text and pred_text else None

    # Check for exact match (normalized)
    exact = (gt_text.lower().strip() == pred_text.lower().strip()) if gt_text and pred_text else False

    results.append({
        "ground_truth": gt_text,
        "prediction": pred_text,
        "rouge": {k: v.fmeasure for k, v in scores.items()} if scores else None,
        "exact_match": exact,
        "empty_pred": len(pred_text) == 0,
    })

    # Progress
    example_num = i + 1
    elapsed = time.time() - start_time
    rate = example_num / elapsed if elapsed > 0 else 0
    eta_min = (total - example_num) / rate / 60 if rate > 0 else 0
    rl = scores["rougeL"].fmeasure if scores else 0
    gt_preview = (gt_text[:60] + "...") if gt_text and len(gt_text) > 60 else (gt_text or "?")
    pred_preview = (pred_text[:60] + "...") if len(pred_text) > 60 else (pred_text or "(empty)")

    print(f"[{example_num}/{total}] ROUGE-L:{rl:.2f} exact:{exact} | {rate:.1f}ex/s | ETA:{eta_min:.0f}min")
    print(f"  GT:   {gt_preview}")
    print(f"  PRED: {pred_preview}")
    print()

print(f"\nDone! {time.time() - start_time:.0f}s total")

In [ ]:
# Cell 9: Compute and Display Metrics

valid = [r for r in results if r["rouge"] is not None]
empty_preds = sum(1 for r in results if r["empty_pred"])

avg_rouge1 = sum(r["rouge"]["rouge1"] for r in valid) / len(valid) if valid else 0
avg_rouge2 = sum(r["rouge"]["rouge2"] for r in valid) / len(valid) if valid else 0
avg_rougeL = sum(r["rouge"]["rougeL"] for r in valid) / len(valid) if valid else 0
exact_matches = sum(1 for r in valid if r["exact_match"])

print("=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)
print(f"  Total examples:       {len(results)}")
print(f"  Valid comparisons:    {len(valid)}")
print(f"  Empty predictions:    {empty_preds}")
print(f"  Exact match rate:     {exact_matches / len(valid):.2%}" if valid else "  N/A")
print(f"  Avg ROUGE-1 (F1):     {avg_rouge1:.4f}")
print(f"  Avg ROUGE-2 (F1):     {avg_rouge2:.4f}")
print(f"  Avg ROUGE-L (F1):     {avg_rougeL:.4f}")
print("=" * 60)

# ROUGE-L distribution
if valid:
    rougeL_scores = sorted([r["rouge"]["rougeL"] for r in valid])
    n = len(rougeL_scores)
    print(f"\n  ROUGE-L distribution:")
    print(f"    Min:    {rougeL_scores[0]:.4f}")
    print(f"    P25:    {rougeL_scores[n // 4]:.4f}")
    print(f"    Median: {rougeL_scores[n // 2]:.4f}")
    print(f"    P75:    {rougeL_scores[3 * n // 4]:.4f}")
    print(f"    Max:    {rougeL_scores[-1]:.4f}")
    print(f"    >0.5:   {sum(1 for s in rougeL_scores if s > 0.5)}/{n} ({sum(1 for s in rougeL_scores if s > 0.5)/n:.0%})")
    print(f"    >0.8:   {sum(1 for s in rougeL_scores if s > 0.8)}/{n} ({sum(1 for s in rougeL_scores if s > 0.8)/n:.0%})")

In [ ]:
# Cell 10: Save Results to Drive
import json
from datetime import datetime

RESULTS_DIR = "/content/drive/MyDrive/fine-tune-mdr/eval_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
report = {
    "total_examples": len(results),
    "valid_comparisons": len(valid),
    "empty_predictions": empty_preds,
    "exact_match_rate": exact_matches / len(valid) if valid else 0,
    "avg_rouge1": avg_rouge1,
    "avg_rouge2": avg_rouge2,
    "avg_rougeL": avg_rougeL,
    "num_examples_evaluated": NUM_EXAMPLES,
    "timestamp": timestamp,
}

out_path = f"{RESULTS_DIR}/{timestamp}.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"Results saved to {out_path}")

# Also save detailed per-example results
details_path = f"{RESULTS_DIR}/{timestamp}_details.json"
with open(details_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Detailed results saved to {details_path}")